In [1]:
%load_ext autoreload
%autoreload 2

import sys, os
# os.environ['PATH']='/projects/bdne/spandey3/tex/texlive/bin/x86_64-linux:'+ os.environ['PATH']
# os.environ['PYTHONPATH']='/projects/bdne/spandey3/tex/texlive/bin/x86_64-linux:'
# os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION']='.98'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
import jax_cosmo.background as bkgrd
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from jax.lib import xla_bridge
platform = xla_bridge.get_backend().platform
import jax
print(jax.local_device_count(), jax.device_count())
jax.config.update('jax_platform_name', platform)
jax.config.update("jax_enable_x64", True)
import jax
# Change the current working directory to the desired path
# os.chdir('/mnt/home/spandey/ceph/GODMAX/src/')
import matplotlib
import matplotlib.pyplot as pl
# set latex to false:
pl.rcParams['text.usetex'] = False
import pathlib
curr_path = pathlib.Path().absolute()
abs_path_data = os.path.abspath(curr_path / "../../data/") 
abs_path_src = os.path.abspath(curr_path / "../../src/") 
abs_path_results = os.path.abspath(curr_path / "../../results/") 
sys.path.append((curr_path))
sys.path.append((abs_path_data))
sys.path.append((abs_path_results))
sys.path.append(abs_path_src)
import numpyro
numpyro.set_platform("gpu")
numpyro.enable_x64()
from jax import config
config.update("jax_enable_x64", True)
import scipy.interpolate as interp
import pickle as pk
import numpy as np
import jax.numpy as jnp
import colossus 
from jax import vmap, grad, pmap
import matplotlib.pyplot as pl
pl.rc('text', usetex=True)
import gc
# Palatino
# pl.rc('font', family='DejaVu Sans')
import ast
import yaml

# nside = int(ast.literal_eval(sys.argv[1]))
# print('nside: ', nside)
# jdevice = int(ast.literal_eval(sys.argv[2]))
# print('jdevice: ', jdevice)
# Ndevices = int(ast.literal_eval(sys.argv[3]))
# print('Ndevices: ', Ndevices)
nside = 2048
jdevice = 0
Ndevices = 8




/tmp/ipykernel_230461/3702773108.py:12: DeprecationWarning: jax.lib.xla_bridge.get_backend is deprecated; use jax.extend.backend.get_backend.
  platform = xla_bridge.get_backend().platform


1 1


/mnt/home/spandey/miniconda3/envs/ili-sbi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import numpy as np

def split_array_power_ratio(arr, num_parts, power_val=0.5):
    n_total = arr.shape[0]
    j_values = np.arange(1, num_parts + 1)
    sqrt_j = j_values**power_val
    sum_of_sqrts = np.sum(sqrt_j)
    ideal_s1 = n_total / sum_of_sqrts
    ideal_sizes = ideal_s1 * sqrt_j
    int_sizes = np.floor(ideal_sizes).astype(int)
    remainder = n_total - np.sum(int_sizes)
    fractional_parts = ideal_sizes - int_sizes
    indices_to_increment = np.argsort(fractional_parts)[-remainder:]
    int_sizes[indices_to_increment] += 1
    split_indices = np.cumsum(int_sizes)[:-1]
    return np.split(arr, split_indices)




In [3]:
# argsort_split = split_array_power_ratio(argsort, Ndevices)
# argsort_here = argsort_split[jdevice]
# # argsort_split
# for i, part in enumerate(argsort_split):
#     print(f"Part {i+1} (size {len(part)}): {part}")



In [4]:
def read_yaml(file_path):
    with open(file_path, 'r') as file:
        data = yaml.safe_load(file)
    return data

def generate_dicts(data):
    sim_params_dict = data.get('sim_params', {})
    halo_params_dict = data.get('halo_params', {})
    analysis_dict = data.get('analysis', {})
    other_params_dict = data.get('other_params', {})
    return sim_params_dict, halo_params_dict, analysis_dict, other_params_dict

yaml_file_path = '/mnt/home/spandey/ceph/GODMAX/param_files/params_default.yaml'
data = read_yaml(yaml_file_path)
sim_params_dict, halo_params_dict, analysis_dict, other_params_dict = generate_dicts(data)

halo_params_dict['rmin'] = 0.0001
halo_params_dict['rmax'] = 10.0
halo_params_dict['nr'] = 126
halo_params_dict['zmin'] = 0.01
halo_params_dict['zmax'] = 4.1
halo_params_dict['nz'] = 127
halo_params_dict['lg10_Mmin'] = 12.0
halo_params_dict['lg10_Mmax'] = 16.0
halo_params_dict['nM'] = 128

from godmax.get_B12_profile import Battaglia_12_16
import helpers.constants as constants
# cosmo_params_dict = {'w0':-1.0 ,'flat': True, 'H0': 69.0, 'Om0': 0.31, 'Ob0': 0.049, 'sigma8':0.81 ,'ns': 0.965}
cosmo_params_dict = {'w0':-1.0 ,'flat': True, 'H0': 67.74, 'Om0': 0.3089, 'Ob0': 0.0486, 'sigma8':0.8159 ,'ns': 0.9667}
# B12_test = Battaglia_12_16({'cosmo':cosmo_params_dict, 'init_power':False}, halo_params_dict)


B12_test = Battaglia_12_16(sim_params_dict={'cosmo':cosmo_params_dict, 'init_power':True}, halo_params_dict=halo_params_dict)

from astropy.io import fits
import h5py as h5
import healpy as hp

fname = '/mnt/ceph/users/abayer/fastpm/halfdome/stampede2_3750Mpch_6144cube/final_res/halos/lightcone_100.hdf5'
with h5.File(fname, 'r') as f:
    # print(f.keys())
    M200c_all = f['halo_mass_m200c'][:]
    z_all = f['redshift'][:]
    pos = f['Position'][:]
    v_all = f['Velocity'][:]

ra_all, dec_all = hp.vec2ang(pos, lonlat=True)

# get the line of sight velocity:
vlos_all = np.sum(v_all * hp.ang2vec(ra_all, dec_all, lonlat=True), axis=1)

zmax = 1.5
indsel = np.where((z_all>0.05) & (z_all<zmax) & (M200c_all<4e15) & (M200c_all>(10**13.0)))[0]
ra_all = ra_all[indsel]
dec_all = dec_all[indsel]
z_all = z_all[indsel]
M200c_all = M200c_all[indsel]
# M200m_all = M200m_all[indsel]
vlos_all = vlos_all[indsel]
print('total number of halos: ', len(M200c_all))

argsort = np.flip(np.argsort(M200c_all))
Nsel = (len(argsort)//Ndevices)*Ndevices
argsort = argsort[:Nsel]
# Ngal_per_device = Nsel//Ndevices
# argsort_here = argsort[jdevice*Ngal_per_device:(jdevice+1)*Ngal_per_device]
argsort_split = split_array_power_ratio(argsort, Ndevices)
argsort_here = argsort_split[jdevice]

ra_all = ra_all[argsort_here]
dec_all = dec_all[argsort_here]
z_all = z_all[argsort_here]
M200c_all = M200c_all[argsort_here]
vlos_all = vlos_all[argsort_here]

print('number of halos: ', len(M200c_all), ', mean log(M200c)', np.mean(np.log10(M200c_all)), ', mean z', np.mean(z_all))
print('log10Mmin: ', np.min(np.log10(M200c_all)), ', log10Mmax: ', np.max(np.log10(M200c_all)))
print('zmin: ', np.min(z_all), ', zmax: ', np.max(z_all))
print('ra min: ', np.min(ra_all), ', ra max: ', np.max(ra_all))
print('dec min: ', np.min(dec_all), ', dec max: ', np.max(dec_all))

import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

import pickle as pk
import jax
import scipy.interpolate as interp
import healpy as hp
import numpy as np
from multiprocessing import Pool, cpu_count
from astropy.io import fits
import jax_cosmo.background as bkgrd
from godmax.get_sim_maps import get_sim_map
import h5py as h5

jax.clear_caches()
# nside = 4096
# nside = 8192
sdir = '/mnt/home/spandey/ceph/GODMAX/notebooks/all_arxiv/mock_gen/maps_halfdome/'
# save_kszmap_fname = sdir + f'tSZ_sim_B12_testv3_nside_{nside}.pkl'
# save_map_fname = sdir + f'kSZ_sim_B16_testv11_nside_{nside}_split_{jdevice}_{Ndevices}.pkl'
save_map_fname = sdir + f'tSZ_sim_B12_testv12_nside_{nside}_split_{jdevice}_{Ndevices}_zmax_{zmax}.pkl'
# if not os.path.exists(save_kszmap_fname):
halo_ra, halo_dec = ra_all, dec_all
halo_z = z_all
halo_m = M200c_all

print('number of halos: ', len(halo_m))
M_all = halo_m
ra_all = halo_ra
dec_all = halo_dec
z_all = halo_z
# vlos_all = np.zeros_like(z_all)
nsel = len(M_all)

if nside == 8192:
    nh_max = 4e3
elif nside == 4096:
    nh_max = 5e4
elif nside == 2048:
    if jdevice == 0:
        nh_max = 5e5
    else:
        nh_max = 8e5
elif nside == 1024:
    nh_max = 2e5
else:
    print('nside not supported')
if nsel > nh_max:
    num_chunks = int(np.ceil(nsel / nh_max))
else:
    num_chunks = 1




total number of halos:  22242936
number of halos:  1364095 , mean log(M200c) 13.92587531765546 , mean z 0.7646463
log10Mmin:  13.729280078938626 , log10Mmax:  15.41060190392757
zmin:  0.05006075 , zmax:  1.4999993
ra min:  0.00023370826 , ra max:  359.99985
dec min:  -89.90721 , dec max:  89.89347
number of halos:  1364095


In [5]:
map_test = np.zeros(12*nside**2, dtype=np.float32)
from tqdm import tqdm

# MEMORY-OPTIMIZED FUNCTIONS
def process_halo(args):
    """Memory-optimized halo processing function"""
    jhalo, ra_all_np, dec_all_np, z_all_chunk, M_all_chunk, vlos_all_chunk, halo_cat_R200c_np, halo_cat_DV_np, max_paint_R200c_factor, nside_local, pixel_dtype = args
    
    vec = hp.ang2vec(ra_all_np[jhalo], dec_all_np[jhalo], lonlat=True)
    nearby_angle = max_paint_R200c_factor * halo_cat_R200c_np[jhalo] / halo_cat_DV_np[jhalo]
    nearby_pix = hp.query_disc(nside_local, vec, nearby_angle)
    
    if len(nearby_pix) == 0:
        return None  # Handle empty results
    
    # Use more efficient data types
    nearby_pix = np.asarray(nearby_pix, dtype=pixel_dtype)
    nearby_ra, nearby_dec = hp.pix2ang(nside_local, nearby_pix, lonlat=True)
    
    # Compute distances more efficiently using vectorized haversine
    ra1, dec1 = np.radians(ra_all_np[jhalo]), np.radians(dec_all_np[jhalo])
    ra2, dec2 = np.radians(nearby_ra), np.radians(nearby_dec)
    
    # Vectorized haversine formula
    dra = ra1 - ra2
    ddec = dec1 - dec2
    a = np.sin(ddec/2)**2 + np.cos(dec1) * np.cos(dec2) * np.sin(dra/2)**2
    theta = 2 * np.arcsin(np.sqrt(a))
    
    distances = (halo_cat_DV_np[jhalo] * theta).astype(np.float32)
    
    # Return only essential data (no redundant arrays)
    return (nearby_pix, distances, jhalo, len(nearby_pix))

def concatenate_results(results, M_all_chunk, z_all_chunk, vlos_all_chunk, halo_cat_DV_np, halo_cat_R200c_np, max_paint_R200c_factor, pixel_dtype):
    """Memory-efficient concatenation of results"""
    if not results:
        return None
        
    # Pre-compute lengths for each part
    lengths = np.array([result[3] for result in results], dtype=np.int32)
    total_length = lengths.sum()
    
    # Pre-allocate arrays with appropriate dtypes
    nearby_pix_all = np.empty(total_length, dtype=pixel_dtype)
    distances_pix_all = np.empty(total_length, dtype=np.float32)
    halo_indices = np.empty(total_length, dtype=np.int32)
    
    # Calculate start and end indices more efficiently
    end_ind_all = np.cumsum(lengths)
    start_ind_all = np.concatenate([[0], end_ind_all[:-1]])
    
    # Use array slicing for assignment
    for i, (start, end, result) in enumerate(zip(start_ind_all, end_ind_all, results)):
        nearby_pix_all[start:end] = result[0]
        distances_pix_all[start:end] = result[1]
        halo_indices[start:end] = result[2]  # Store halo index instead of repeated values
    
    # Create derived arrays more efficiently (avoid redundant storage)
    halo_start_indices = np.array([result[2] for result in results], dtype=np.int32)
    logM_ind_all = np.log(M_all_chunk[halo_indices], dtype=np.float32)
    z_ind_all = z_all_chunk[halo_indices].astype(np.float32)
    vlos_ind_all = vlos_all_chunk[halo_indices].astype(np.float32)
    ang_distance_all = halo_cat_DV_np[halo_start_indices].astype(np.float32)
    rp_max_all = (max_paint_R200c_factor * halo_cat_R200c_np[halo_start_indices]).astype(np.float32)
    
    return (nearby_pix_all, distances_pix_all, start_ind_all, end_ind_all, 
            logM_ind_all, z_ind_all, vlos_ind_all, ang_distance_all, rp_max_all)

def process_halos_in_batches(M_all_chunk, ra_all_chunk, dec_all_chunk, z_all_chunk, vlos_all_chunk, 
                           halo_cat_R200c, halo_cat_DA, max_paint_R200c_factor, nside, batch_size=1000):
    """Process halos in batches to reduce peak memory usage"""
    
    # Determine appropriate pixel data type
    pixel_dtype = np.int32 if nside <= 8192 else np.int64
    
    # Convert to memory-efficient data types
    ra_all_np = np.clip(np.array(ra_all_chunk, dtype=np.float32), 0.01, 359.99)
    dec_all_np = np.clip(np.array(dec_all_chunk, dtype=np.float32), -89.99, 89.99)
    z_all_np = np.array(z_all_chunk, dtype=np.float32)
    halo_cat_R200c_np = np.array(halo_cat_R200c, dtype=np.float32)
    halo_cat_DV_np = np.array(halo_cat_DA, dtype=np.float32)
    halo_vlos_np = np.array(vlos_all_chunk, dtype=np.float32)
    M_all_np = np.array(M_all_chunk, dtype=np.float32)
    
    n_halos = len(z_all_chunk)
    all_results = []
    
    for batch_start in range(0, n_halos, batch_size):
        batch_end = min(batch_start + batch_size, n_halos)
        print(f"Processing batch {batch_start//batch_size + 1}/{(n_halos-1)//batch_size + 1}")
        
        # Prepare arguments for this batch
        batch_args = []
        for jhalo in range(batch_start, batch_end):
            args = (jhalo, ra_all_np, dec_all_np, z_all_np, M_all_np, halo_vlos_np, 
                   halo_cat_R200c_np, halo_cat_DV_np, max_paint_R200c_factor, nside, pixel_dtype)
            batch_args.append(args)
        
        # Process batch
        with Pool(cpu_count()) as pool:
            batch_results = pool.map(process_halo, batch_args)
        
        # Filter out None results
        batch_results = [r for r in batch_results if r is not None]
        
        if batch_results:
            # Get batch-specific arrays for concatenation
            batch_M = M_all_np[batch_start:batch_end]
            batch_z = z_all_np[batch_start:batch_end] 
            batch_vlos = halo_vlos_np[batch_start:batch_end]
            batch_DV = halo_cat_DV_np[batch_start:batch_end]
            batch_R200c = halo_cat_R200c_np[batch_start:batch_end]
            
            # Adjust halo indices to be relative to the full array
            batch_results_adjusted = []
            for result in batch_results:
                pix, dist, halo_idx, n_pix = result
                adjusted_idx = halo_idx  # Keep original index
                batch_results_adjusted.append((pix, dist, adjusted_idx, n_pix))
            
            # Concatenate batch results
            batch_data = concatenate_results(batch_results_adjusted, M_all_np, z_all_np, halo_vlos_np, 
                                           halo_cat_DV_np, halo_cat_R200c_np, max_paint_R200c_factor, pixel_dtype)
            
            if batch_data is not None:
                all_results.append(batch_data)
        
        # Clear batch data
        del batch_results, batch_args
        gc.collect()
    
    if not all_results:
        return None
        
    # Final concatenation of all batches
    return final_concatenate_batches(all_results, pixel_dtype)

def final_concatenate_batches(all_results, pixel_dtype):
    """Final concatenation of batch results"""
    total_pix = sum(len(result[0]) for result in all_results)
    total_halos = sum(len(result[2]) for result in all_results)
    
    # Pre-allocate final arrays
    final_nearby_pix = np.empty(total_pix, dtype=pixel_dtype)
    final_distances = np.empty(total_pix, dtype=np.float32)
    final_logM = np.empty(total_pix, dtype=np.float32)
    final_z = np.empty(total_pix, dtype=np.float32)
    final_vlos = np.empty(total_pix, dtype=np.float32)
    final_start_ind = np.empty(total_halos, dtype=np.int32)
    final_end_ind = np.empty(total_halos, dtype=np.int32)
    final_ang_dist = np.empty(total_halos, dtype=np.float32)
    final_rp_max = np.empty(total_halos, dtype=np.float32)
    
    pix_offset = 0
    halo_offset = 0
    
    for result in all_results:
        nearby_pix_all, distances_pix_all, start_ind_all, end_ind_all, logM_ind_all, z_ind_all, vlos_ind_all, ang_distance_all, rp_max_all = result
        
        n_pix_batch = len(nearby_pix_all)
        n_halo_batch = len(start_ind_all)
        
        # Copy pixel data
        final_nearby_pix[pix_offset:pix_offset + n_pix_batch] = nearby_pix_all
        final_distances[pix_offset:pix_offset + n_pix_batch] = distances_pix_all
        final_logM[pix_offset:pix_offset + n_pix_batch] = logM_ind_all
        final_z[pix_offset:pix_offset + n_pix_batch] = z_ind_all
        final_vlos[pix_offset:pix_offset + n_pix_batch] = vlos_ind_all
        
        # Copy halo data with offset adjustment
        final_start_ind[halo_offset:halo_offset + n_halo_batch] = start_ind_all + pix_offset
        final_end_ind[halo_offset:halo_offset + n_halo_batch] = end_ind_all + pix_offset
        final_ang_dist[halo_offset:halo_offset + n_halo_batch] = ang_distance_all
        final_rp_max[halo_offset:halo_offset + n_halo_batch] = rp_max_all
        
        pix_offset += n_pix_batch
        halo_offset += n_halo_batch
    
    return (final_nearby_pix, final_distances, final_start_ind, final_end_ind,
            final_logM, final_z, final_vlos, final_ang_dist, final_rp_max)

# MAIN PROCESSING LOOP WITH OPTIMIZATIONS
for i in tqdm(range(num_chunks)):
    if i == num_chunks - 1:
        M_all_chunk = M_all[int(i*nh_max):]
        ra_all_chunk = ra_all[int(i*nh_max):]
        dec_all_chunk = dec_all[int(i*nh_max):]
        z_all_chunk = z_all[int(i*nh_max):]
        vlos_all_chunk = vlos_all[int(i*nh_max):]
    else:
        M_all_chunk = M_all[int(i*nh_max):int((i+1)*nh_max)]
        ra_all_chunk = ra_all[int(i*nh_max):int((i+1)*nh_max)]
        dec_all_chunk = dec_all[int(i*nh_max):int((i+1)*nh_max)]
        z_all_chunk = z_all[int(i*nh_max):int((i+1)*nh_max)]
        vlos_all_chunk = vlos_all[int(i*nh_max):int((i+1)*nh_max)]

    mock_params_dict = {}
    mock_params_dict['nside'] = nside
    mock_params_dict['get_ymap'] = True
    mock_params_dict['smooth_profiles'] = True

    halo_cat_scale_fac = 1./(1. + z_all_chunk)
    halo_cat_rho_c_z = constants.RHO_CRIT_0_KPC3 * bkgrd.Esqr(B12_test.cosmo_jax, halo_cat_scale_fac) * 1e9
    mdef_delta = 200
    halo_cat_rho_treshold = mdef_delta * halo_cat_rho_c_z
    halo_cat_R200c = (M_all_chunk * 3.0 / 4.0 / jnp.pi / halo_cat_rho_treshold)**(1.0 / 3.0)
    halo_cat_DA = bkgrd.angular_diameter_distance(B12_test.cosmo_jax, halo_cat_scale_fac)
    max_paint_R200c_factor = 3.

    # Determine batch size based on available memory and nside
    # if nside >= 4096:
    #     batch_size = 500
    # elif nside >= 2048:
    #     batch_size = 1000
    # else:
    #     batch_size = 2000
    batch_size = int(nh_max//2)
    # Process halos with memory optimization
    print('processing batches')
    result = process_halos_in_batches(
        M_all_chunk, ra_all_chunk, dec_all_chunk, z_all_chunk, vlos_all_chunk,
        halo_cat_R200c, halo_cat_DA, max_paint_R200c_factor, nside, batch_size
    )
    print('finished processing batches')

    if result is not None:
        nearby_pix_all, distances_pix_all, start_ind_all, end_ind_all, logM_ind_all, z_ind_all, vlos_ind_all, ang_distance_all, rp_max_all = result

        # Populate mock_params_dict efficiently
        mock_params_dict['halo_z'] = jnp.array(z_all_chunk, dtype=jnp.float32)
        mock_params_dict['halo_ra'] = jnp.array(ra_all_chunk, dtype=jnp.float32)
        mock_params_dict['halo_dec'] = jnp.array(dec_all_chunk, dtype=jnp.float32)
        mock_params_dict['halo_M'] = jnp.array(M_all_chunk, dtype=jnp.float32)
        mock_params_dict['halo_vlos'] = jnp.array(vlos_all_chunk, dtype=jnp.float32)

        mock_params_dict['nearby_pix_all'] = jnp.array(nearby_pix_all)
        mock_params_dict['pix_prop_all'] = jnp.array([np.log(distances_pix_all), z_ind_all, logM_ind_all, vlos_ind_all]).T
        mock_params_dict['ang_distance_all'] = jnp.array(ang_distance_all)
        mock_params_dict['rp_max_all'] = jnp.array(rp_max_all)
        mock_params_dict['start_ind'] = jnp.array(start_ind_all, dtype=jnp.int32)
        mock_params_dict['end_ind'] = jnp.array(end_ind_all, dtype=jnp.int32)

        # Generate mock map
        print('generating mock map')
        mock_map_test = get_sim_map(sim_params_dict, halo_params_dict, analysis_dict, other_params_dict, mock_params_dict, Profiles_obj=B12_test)
        map_test += np.array(np.nan_to_num(mock_map_test.ymap_final), dtype=np.float32)
        print('finished generating mock map')

    # Clear memory
    del result, mock_params_dict
    if 'mock_map_test' in locals():
        del mock_map_test
    gc.collect()

# Save results
saved = {'map_test': map_test}
pk.dump(saved, open(save_map_fname, 'wb'))




  0%|          | 0/3 [00:00<?, ?it/s]

processing batches
Processing batch 1/2
Processing batch 2/2
finished processing batches
generating mock map


 33%|███▎      | 1/3 [01:01<02:03, 61.57s/it]

finished generating mock map
processing batches
Processing batch 1/2
Processing batch 2/2
finished processing batches
generating mock map


 67%|██████▋   | 2/3 [01:52<00:55, 55.11s/it]

finished generating mock map
processing batches
Processing batch 1/2
Processing batch 2/2
finished processing batches
generating mock map


100%|██████████| 3/3 [02:39<00:00, 53.08s/it]

finished generating mock map


In [1]:
#!/usr/bin/env python
"""
Distributed tSZ map generation using JAX distributed across multiple nodes and GPUs.
This code automatically handles node/GPU distribution using JAX's distributed module.
"""
%load_ext autoreload
%autoreload 2
import sys
import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'

# ==============================================================================
# JAX Multi-Node/Multi-GPU Initialization - MUST BE FIRST
# ==============================================================================
import jax
import jax.distributed
# jax.distributed.initialize()

from jax.lib import xla_bridge
platform = xla_bridge.get_backend().platform
print(f"JAX process {jax.process_index()}/{jax.process_count()} initialized on platform: {platform}")
print(f"Local device count: {jax.local_device_count()}, Total device count: {jax.device_count()}")

jax.config.update('jax_platform_name', platform)
jax.config.update("jax_enable_x64", True)

import numpyro
numpyro.set_platform("gpu")
numpyro.enable_x64()

import numpy as np
import jax.numpy as jnp
from jax import pmap, jit
import healpy as hp
import yaml
import ast
from tqdm import tqdm
import pickle as pk
import h5py as h5
from multiprocessing import Pool, cpu_count
import warnings
import gc
import pathlib
from functools import partial
from typing import Dict, Tuple, Any

# Suppress warnings
warnings.filterwarnings("ignore")

# Import custom modules
import pathlib
curr_path = pathlib.Path().absolute()
abs_path_data = os.path.abspath(curr_path / "../../data/") 
abs_path_src = os.path.abspath(curr_path / "../../src/") 
abs_path_results = os.path.abspath(curr_path / "../../results/") 
sys.path.append((curr_path))
sys.path.append((abs_path_data))
sys.path.append((abs_path_results))
sys.path.append(abs_path_src)
import jax_cosmo.background as bkgrd
from godmax.get_B12_profile import Battaglia_12_16
import helpers.constants as constants
# from godmax.get_sim_maps import get_sim_map

# ==============================================================================
# Configuration Functions
# ==============================================================================

def read_yaml(file_path: str) -> dict:
    """Read YAML configuration file."""
    with open(file_path, 'r') as file:
        data = yaml.safe_load(file)
    return data

def generate_dicts(data: dict) -> Tuple[dict, dict, dict, dict]:
    """Generate parameter dictionaries from YAML data."""
    sim_params_dict = data.get('sim_params', {})
    halo_params_dict = data.get('halo_params', {})
    analysis_dict = data.get('analysis', {})
    other_params_dict = data.get('other_params', {})
    return sim_params_dict, halo_params_dict, analysis_dict, other_params_dict

def setup_paths() -> Tuple[pathlib.Path, ...]:
    """Setup and return necessary paths."""
    curr_path = pathlib.Path().absolute()
    abs_path_data = os.path.abspath(curr_path / "../../data/")
    abs_path_src = os.path.abspath(curr_path / "../../src/")
    abs_path_results = os.path.abspath(curr_path / "../../results/")
    
    # Add to path
    for path in [str(curr_path), abs_path_data, abs_path_results, abs_path_src]:
        if path not in sys.path:
            sys.path.append(path)
    
    return curr_path, abs_path_data, abs_path_src, abs_path_results

# ==============================================================================
# Data Loading and Distribution Functions
# ==============================================================================

def load_and_filter_catalog(fname: str, zmin: float = 0.05, zmax: float = 1.5, 
                           Mmin: float = 1e13, Mmax: float = 4e15) -> Tuple[np.ndarray, ...]:
    """Load and filter halo catalog."""
    with h5.File(fname, 'r') as f:
        M200c_all = f['halo_mass_m200c'][:]
        z_all = f['redshift'][:]
        pos = f['Position'][:]
        v_all = f['Velocity'][:]
    
    # Convert positions to RA/Dec
    ra_all, dec_all = hp.vec2ang(pos, lonlat=True)
    
    # Calculate line-of-sight velocities
    vlos_all = np.sum(v_all * hp.ang2vec(ra_all, dec_all, lonlat=True), axis=1)
    
    # Filter halos
    mask = (z_all > zmin) & (z_all < zmax) & (M200c_all > Mmin) & (M200c_all < Mmax)
    
    return (ra_all[mask], dec_all[mask], z_all[mask], 
            M200c_all[mask], vlos_all[mask])

def distribute_halos_across_nodes(ra_all: np.ndarray, dec_all: np.ndarray, 
                                 z_all: np.ndarray, M200c_all: np.ndarray, 
                                 vlos_all: np.ndarray, num_nodes: int) -> Tuple[np.ndarray, ...]:
    """Distribute halos across nodes, sorted by mass."""
    # Sort by mass (descending)
    argsort = np.flip(np.argsort(M200c_all))
    
    # Apply sorting
    ra_all = ra_all[argsort]
    dec_all = dec_all[argsort]
    z_all = z_all[argsort]
    M200c_all = M200c_all[argsort]
    vlos_all = vlos_all[argsort]
    
    # Ensure divisibility by number of nodes
    n_halos = len(M200c_all)
    n_halos_trimmed = (n_halos // num_nodes) * num_nodes
    
    if n_halos_trimmed < n_halos:
        print(f"Trimming {n_halos - n_halos_trimmed} halos for even distribution")
        ra_all = ra_all[:n_halos_trimmed]
        dec_all = dec_all[:n_halos_trimmed]
        z_all = z_all[:n_halos_trimmed]
        M200c_all = M200c_all[:n_halos_trimmed]
        vlos_all = vlos_all[:n_halos_trimmed]
    
    # Split data for all nodes
    ra_splits = np.array_split(ra_all, num_nodes)
    dec_splits = np.array_split(dec_all, num_nodes)
    z_splits = np.array_split(z_all, num_nodes)
    m_splits = np.array_split(M200c_all, num_nodes)
    vlos_splits = np.array_split(vlos_all, num_nodes)
    
    return list(zip(ra_splits, dec_splits, z_splits, m_splits, vlos_splits))

# ==============================================================================
# CPU Pre-processing Functions
# ==============================================================================

# def process_halo_cpu(args: Tuple) -> Tuple[np.ndarray, ...]:
#     """Process a single halo on CPU to find nearby pixels."""
#     (jhalo, ra_arr, dec_arr, r200c_arr, da_arr, m_arr, 
#      z_arr, vlos_arr, nside, max_paint_factor) = args
    
#     # Find nearby pixels
#     vec = hp.ang2vec(ra_arr[jhalo], dec_arr[jhalo], lonlat=True)
#     nearby_angle = max_paint_factor * r200c_arr[jhalo] / da_arr[jhalo]
#     nearby_pix = hp.query_disc(nside, vec, nearby_angle)
#     nearby_ra, nearby_dec = hp.pix2ang(nside, nearby_pix, lonlat=True)
    
#     # Calculate physical distances using Haversine formula
#     def haversine(theta):
#         return np.sin(theta / 2.) ** 2.
    
#     ra1, dec1 = np.deg2rad(ra_arr[jhalo]), np.deg2rad(dec_arr[jhalo])
#     ra2, dec2 = np.deg2rad(nearby_ra), np.deg2rad(nearby_dec)
#     theta = 2. * np.arcsin(np.sqrt(
#         haversine(dec1 - dec2) + np.cos(dec1) * np.cos(dec2) * haversine(ra1 - ra2)
#     ))
#     physical_distances = da_arr[jhalo] * theta
    
#     num_pix = len(nearby_pix)
#     return (
#         np.array(nearby_pix, dtype=np.int32),
#         np.array(physical_distances, dtype=np.float64),
#         np.full(num_pix, np.log(m_arr[jhalo]), dtype=np.float64),
#         np.full(num_pix, z_arr[jhalo], dtype=np.float64),
#         np.full(num_pix, vlos_arr[jhalo], dtype=np.float64),
#         np.full(num_pix, da_arr[jhalo], dtype=np.float64),
#         np.full(num_pix, max_paint_factor * r200c_arr[jhalo], dtype=np.float64)
#     )



def prepare_chunk_for_gpu(M_chunk: np.ndarray, ra_chunk: np.ndarray, 
                          dec_chunk: np.ndarray, z_chunk: np.ndarray, 
                          vlos_chunk: np.ndarray, B12_test: Any, 
                          nside: int, n_cpus: int = None) -> Dict:
    """Prepare a chunk of halos for GPU processing."""
    if len(M_chunk) == 0:
        return None
    
    # Calculate halo properties
    scale_fac = 1. / (1. + z_chunk)
    rho_c_z = constants.RHO_CRIT_0_KPC3 * bkgrd.Esqr(B12_test.cosmo_jax, scale_fac) * 1e9
    rho_threshold = 200 * rho_c_z
    R200c = np.array((M_chunk * 3.0 / (4.0 * np.pi * rho_threshold))**(1.0 / 3.0))
    DA = np.array(bkgrd.angular_diameter_distance(B12_test.cosmo_jax, scale_fac))
    max_paint_R200c_factor = 3.0
    
    # Clip coordinates to valid ranges
    ra_chunk_clipped = np.clip(ra_chunk, 0.01, 359.99)
    dec_chunk_clipped = np.clip(dec_chunk, -89.99, 89.99)

    def process_halo_cpu(jhalo) -> Tuple[np.ndarray, ...]:
        """Process a single halo on CPU to find nearby pixels."""
        # (jhalo, ra_arr, dec_arr, r200c_arr, da_arr, m_arr, 
        # z_arr, vlos_arr, nside, max_paint_factor) = args
        
        # Find nearby pixels
        vec = hp.ang2vec(ra_chunk_clipped[jhalo], dec_chunk_clipped[jhalo], lonlat=True)
        nearby_angle = max_paint_factor * R200c[jhalo] / DA[jhalo]
        nearby_pix = hp.query_disc(nside, vec, nearby_angle)
        nearby_ra, nearby_dec = hp.pix2ang(nside, nearby_pix, lonlat=True)
        
        # Calculate physical distances using Haversine formula
        def haversine(theta):
            return np.sin(theta / 2.) ** 2.
        
        ra1, dec1 = np.deg2rad(ra_arr[jhalo]), np.deg2rad(dec_arr[jhalo])
        ra2, dec2 = np.deg2rad(nearby_ra), np.deg2rad(nearby_dec)
        theta = 2. * np.arcsin(np.sqrt(
            haversine(dec1 - dec2) + np.cos(dec1) * np.cos(dec2) * haversine(ra1 - ra2)
        ))
        physical_distances = da_arr[jhalo] * theta
        
        num_pix = len(nearby_pix)
        return (
            np.array(nearby_pix, dtype=np.int32),
            np.array(physical_distances, dtype=np.float64),
            np.full(num_pix, np.log(M_chunk[jhalo]), dtype=np.float64),
            np.full(num_pix, z_chunk[jhalo], dtype=np.float64),
            np.full(num_pix, vlos_chunk[jhalo], dtype=np.float64),
            np.full(num_pix, DA[jhalo], dtype=np.float64),
            np.full(num_pix, max_paint_factor * R200c[jhalo], dtype=np.float64)
        )

    # Prepare arguments for multiprocessing
    # pool_args = [
    #     (j, ra_chunk_clipped, dec_chunk_clipped, np.array(R200c), 
    #      np.array(DA), M_chunk, z_chunk, vlos_chunk, nside, max_paint_R200c_factor)
    #     for j in range(len(z_chunk))
    # ]
    
    # Process halos in parallel on CPU
    if n_cpus is None:
        n_cpus = cpu_count()
    print(f"Using {n_cpus} CPU cores for preprocessing")
    with Pool(processes=n_cpus) as pool:
        # results = pool.map(process_halo_cpu, pool_args)
        results = pool.map(process_halo_cpu, range(len(z_chunk)))

    # Concatenate results
    (nearby_pix_all, distances_pix_all, start_ind_all, end_ind_all, 
     logM_ind_all, z_ind_all, vlos_ind_all, ang_distance_all, rp_max_all) = concatenate_halo_results(results)
    
    if len(nearby_pix_all) == 0:
        return None
    
    return {
        'halo_z': jnp.array(z_chunk),
        'halo_ra': jnp.array(ra_chunk_clipped),
        'halo_dec': jnp.array(dec_chunk_clipped),
        'halo_M': jnp.array(M_chunk),
        'halo_vlos': jnp.array(vlos_chunk),
        'nearby_pix_all': jnp.array(nearby_pix_all),
        'pix_prop_all': jnp.array([
            np.log(distances_pix_all), z_ind_all, 
            logM_ind_all, vlos_ind_all
        ]).T,
        'start_ind': jnp.int32(start_ind_all),
        'end_ind': jnp.int32(end_ind_all),
        'ang_distance_all': jnp.array(ang_distance_all)[start_ind_all],
        'rp_max_all': jnp.array(rp_max_all)[start_ind_all],
        'nside': nside,
        'get_ymap': True,
        'smooth_profiles': True
    }

def get_chunk_size(nside: int) -> int:
    """Get optimal chunk size based on nside."""
    chunk_sizes = {
        512: 2e6,
        1024: 1e5,
        2048: 5e5,
        4096: 5e4,
        8192: 4e3
    }
    return int(chunk_sizes.get(nside, 1e5))




/tmp/ipykernel_1873416/1588699921.py:20: DeprecationWarning: jax.lib.xla_bridge.get_backend is deprecated; use jax.extend.backend.get_backend.
  platform = xla_bridge.get_backend().platform


JAX process 0/1 initialized on platform: gpu
Local device count: 2, Total device count: 2


/mnt/home/spandey/miniconda3/envs/ili-sbi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==============================================================================
# Main Execution
# ==============================================================================

# def main():
# Parse command line arguments
# if len(sys.argv) != 2:
#     if jax.process_index() == 0:
#         print(f"Usage: python {sys.argv[0]} <nside>")
#         print("Example: python tsz_map_distributed.py 2048")
#     sys.exit(1)

# nside = int(ast.literal_eval(sys.argv[1]))
nside = 1024

# Get node rank and world size from JAX distributed
node_rank = jax.process_index()
num_nodes = jax.process_count()
n_gpus_local = jax.local_device_count()

if node_rank == 0:
    print(f"="*60)
    print(f"Starting distributed tSZ map generation")
    print(f"NSIDE: {nside}")
    print(f"Total nodes: {num_nodes}")
    print(f"GPUs per node: {n_gpus_local}")
    print(f"Total GPUs: {jax.device_count()}")
    print(f"="*60)

# Setup paths
setup_paths()

# Load configuration
yaml_file_path = '/mnt/home/spandey/ceph/GODMAX/param_files/params_default.yaml'
data = read_yaml(yaml_file_path)
sim_params_dict, halo_params_dict, analysis_dict, other_params_dict = generate_dicts(data)

# Update halo parameters
halo_params_dict.update({
    'rmin': 0.0001, 'rmax': 10.0, 'nr': 126,
    'zmin': 0.01, 'zmax': 4.1, 'nz': 127,
    'lg10_Mmin': 12.0, 'lg10_Mmax': 16.0, 'nM': 128
})

# Setup cosmology
cosmo_params_dict = {
    'w0': -1.0, 'flat': True, 'H0': 67.74,
    'Om0': 0.3089, 'Ob0': 0.0486,
    'sigma8': 0.8159, 'ns': 0.9667
}

# Initialize Battaglia profile
B12_test = Battaglia_12_16(
    sim_params_dict={'cosmo': cosmo_params_dict, 'init_power': True},
    halo_params_dict=halo_params_dict
)

# Load and distribute data
zmax = 0.5
# if node_rank == 0:
print("Loading halo catalog...")
fname = '/mnt/ceph/users/abayer/fastpm/halfdome/stampede2_3750Mpch_6144cube/final_res/halos/lightcone_100.hdf5'
ra_all, dec_all, z_all, M200c_all, vlos_all = load_and_filter_catalog(
    fname, zmin=0.05, zmax=zmax, Mmin=1e13, Mmax=4e15
)

print(f"Total halos after filtering: {len(M200c_all):,}")

# Distribute halos across nodes
data_splits = distribute_halos_across_nodes(
    ra_all, dec_all, z_all, M200c_all, vlos_all, num_nodes
)
# else:
#     data_splits = None

# Broadcast data from rank 0 to all other ranks
# data_splits = jax.experimental.multihost_utils.broadcast_one_to_all(data_splits)

ra_node, dec_node, z_node, M200c_node, vlos_node = data_splits[node_rank]

print(f"Node {node_rank}: Processing {len(M200c_node):,} halos")
print(f"Node {node_rank}: log10(M) range: [{np.min(np.log10(M200c_node)):.2f}, "
        f"{np.max(np.log10(M200c_node)):.2f}]")
print(f"Node {node_rank}: z range: [{np.min(z_node):.2f}, {np.max(z_node):.2f}]")

# Initialize output map
map_node = np.zeros(12 * nside**2, dtype=np.float32)

# Determine chunk size
chunk_size = get_chunk_size(nside)
num_chunks = int(np.ceil(len(M200c_node) / chunk_size))

print(f"Node {node_rank}: Processing {num_chunks} chunks of size {chunk_size:.0f}")

# # Define pmapped function for multi-GPU execution
# @partial(pmap, in_axes=(None, None, None, None, 
#                         {'start_ind': 0, 'end_ind': 0, 
#                         'ang_distance_all': 0, 'rp_max_all': 0,
#                         'halo_z': 0, 'halo_ra': 0, 'halo_dec': 0,
#                         'halo_M': 0, 'halo_vlos': 0,
#                         'nearby_pix_all': None, 'pix_prop_all': None,
#                         'nside': None, 'get_ymap': None, 'smooth_profiles': None},
#                         None),
#         static_broadcasted_argnums=(0, 1, 2, 3, 5))
# def get_sim_map_pmap(sim_params, halo_params, analysis, other_params, mock_params, profile):
#     return get_sim_map(sim_params, halo_params, analysis, other_params, mock_params, profile)



Starting distributed tSZ map generation
NSIDE: 1024
Total nodes: 1
GPUs per node: 2
Total GPUs: 2
Loading halo catalog...
Total halos after filtering: 3,178,043
Node 0: Processing 3,178,043 halos
Node 0: log10(M) range: [13.00, 15.41]
Node 0: z range: [0.05, 0.50]
Node 0: Processing 32 chunks of size 100000


In [3]:
# chunk_idx = 0
# start_idx = chunk_idx * chunk_size
# end_idx = min((chunk_idx + 1) * chunk_size, len(M200c_node))
# M_chunk = M200c_node[start_idx:end_idx]
# ra_chunk = ra_node[start_idx:end_idx]
# dec_chunk = dec_node[start_idx:end_idx]
# z_chunk = z_node[start_idx:end_idx]
# vlos_chunk = vlos_node[start_idx:end_idx]

            
# # Calculate halo properties
# scale_fac = 1. / (1. + z_chunk)
# rho_c_z = constants.RHO_CRIT_0_KPC3 * bkgrd.Esqr(B12_test.cosmo_jax, scale_fac) * 1e9
# rho_threshold = 200 * rho_c_z
# R200c = np.array((M_chunk * 3.0 / (4.0 * np.pi * rho_threshold))**(1.0 / 3.0))
# DA = np.array(bkgrd.angular_diameter_distance(B12_test.cosmo_jax, scale_fac))
# max_paint_R200c_factor = 3.0

# # Clip coordinates to valid ranges
# ra_chunk_clipped = np.clip(ra_chunk, 0.01, 359.99)
# dec_chunk_clipped = np.clip(dec_chunk, -89.99, 89.99)

# def process_halo_cpu(jhalo) -> Tuple[np.ndarray, ...]:
#     """Process a single halo on CPU to find nearby pixels."""
#     # (jhalo, ra_arr, dec_arr, r200c_arr, da_arr, m_arr, 
#     # z_arr, vlos_arr, nside, max_paint_factor) = args
    
#     # Find nearby pixels
#     vec = hp.ang2vec(ra_chunk_clipped[jhalo], dec_chunk_clipped[jhalo], lonlat=True)
#     nearby_angle = max_paint_R200c_factor * R200c[jhalo] / DA[jhalo]
#     nearby_pix = hp.query_disc(nside, vec, nearby_angle)
#     nearby_ra, nearby_dec = hp.pix2ang(nside, nearby_pix, lonlat=True)
    
#     # Calculate physical distances using Haversine formula
#     def haversine(theta):
#         return np.sin(theta / 2.) ** 2.
    
#     ra1, dec1 = np.deg2rad(ra_chunk_clipped[jhalo]), np.deg2rad(dec_chunk_clipped[jhalo])
#     ra2, dec2 = np.deg2rad(nearby_ra), np.deg2rad(nearby_dec)
#     theta = 2. * np.arcsin(np.sqrt(
#         haversine(dec1 - dec2) + np.cos(dec1) * np.cos(dec2) * haversine(ra1 - ra2)
#     ))
#     physical_distances = DA[jhalo] * theta

#     num_pix = len(nearby_pix)
#     return (
#         np.array(nearby_pix, dtype=np.int32),
#         np.array(physical_distances, dtype=np.float64),
#         np.full(num_pix, np.log(M_chunk[jhalo]), dtype=np.float64),
#         np.full(num_pix, z_chunk[jhalo], dtype=np.float64),
#         np.full(num_pix, vlos_chunk[jhalo], dtype=np.float64),
#         np.full(num_pix, DA[jhalo], dtype=np.float64),
#         np.full(num_pix, max_paint_R200c_factor * R200c[jhalo], dtype=np.float64)
#     )

# # Process halos in parallel on CPU
# # if n_cpus is None:
# n_cpus = cpu_count()
# print(f"Using {n_cpus} CPU cores for preprocessing")
# with Pool(processes=n_cpus) as pool:
#     # results = pool.map(process_halo_cpu, pool_args)
#     results = pool.map(process_halo_cpu, range(len(z_chunk)))

# # Concatenate results
# (nearby_pix_all, distances_pix_all, start_ind_all, end_ind_all, 
#     logM_ind_all, z_ind_all, vlos_ind_all, ang_distance_all, rp_max_all) = concatenate_halo_results(results)


# mock_params_dict = {
#     'halo_z': jnp.array(z_chunk),
#     'halo_ra': jnp.array(ra_chunk_clipped),
#     'halo_dec': jnp.array(dec_chunk_clipped),
#     'halo_M': jnp.array(M_chunk),
#     'halo_vlos': jnp.array(vlos_chunk),
#     'nearby_pix_all': jnp.array(nearby_pix_all),
#     'pix_prop_all': jnp.array([
#         np.log(distances_pix_all), z_ind_all, 
#         logM_ind_all, vlos_ind_all
#     ]).T,
#     'start_ind': jnp.int32(start_ind_all),
#     'end_ind': jnp.int32(end_ind_all),
#     'ang_distance_all': jnp.array(ang_distance_all)[start_ind_all],
#     'rp_max_all': jnp.array(rp_max_all)[start_ind_all],
#     'nside': nside,
#     'get_ymap': True,
#     'smooth_profiles': True
# }







In [3]:
def concatenate_halo_results(results: list) -> Tuple[np.ndarray, ...]:
    """Concatenate results from parallel halo processing."""
    if not results:
        return (np.array([]), np.array([]), np.array([]), np.array([]), 
                np.array([]), np.array([]), np.array([]), np.array([]), np.array([]))
    
    lengths = np.array([len(r[0]) for r in results])
    total_length = lengths.sum()
    
    if total_length == 0:
        return (np.array([]), np.array([]), np.array([]), np.array([]), 
                np.array([]), np.array([]), np.array([]), np.array([]), np.array([]))
    
    # Calculate indices
    end_ind_all = np.cumsum(lengths)
    start_ind_all = np.zeros_like(end_ind_all)
    start_ind_all[1:] = end_ind_all[:-1]
    
    # Pre-allocate arrays
    nearby_pix_all = np.empty(total_length, dtype=np.int32)
    distances_pix_all = np.empty(total_length, dtype=np.float64)
    logM_ind_all = np.empty(total_length, dtype=np.float64)
    z_ind_all = np.empty(total_length, dtype=np.float64)
    vlos_ind_all = np.empty(total_length, dtype=np.float64)
    ang_distance_all = np.empty(total_length, dtype=np.float64)
    rp_max_all = np.empty(total_length, dtype=np.float64)
    
    # Fill arrays
    for i, res in enumerate(results):
        start, end = start_ind_all[i], end_ind_all[i]
        nearby_pix_all[start:end] = res[0]
        distances_pix_all[start:end] = res[1]
        logM_ind_all[start:end] = res[2]
        z_ind_all[start:end] = res[3]
        vlos_ind_all[start:end] = res[4]
        ang_distance_all[start:end] = res[5]
        rp_max_all[start:end] = res[6]
    
    return (nearby_pix_all, distances_pix_all, start_ind_all, end_ind_all, 
            logM_ind_all, z_ind_all, vlos_ind_all, ang_distance_all, rp_max_all)


In [4]:
# Process chunks
# for chunk_idx in tqdm(range(num_chunks), desc=f"Node {node_rank}", disable=(node_rank != 0)):
for chunk_idx in tqdm(range(1), desc=f"Node {node_rank}", disable=(node_rank != 0)):    
    # Get chunk slice
    start_idx = chunk_idx * chunk_size
    end_idx = min((chunk_idx + 1) * chunk_size, len(M200c_node))
    print(start_idx, end_idx)
    M_chunk_all = M200c_node[start_idx:end_idx]
    ra_chunk_all = ra_node[start_idx:end_idx]
    dec_chunk_all = dec_node[start_idx:end_idx]
    z_chunk_all = z_node[start_idx:end_idx]
    vlos_chunk_all = vlos_node[start_idx:end_idx]

    # num_halos_pmap = (num_halos_chunk // n_gpus_local) * n_gpus_local    
    num_halos_per_gpu = len(M_chunk_all) // n_gpus_local
    mock_params_dict_all_gpus = {}

    nelem_all = []
    for jgpu in range(n_gpus_local):
        start_gpu = jgpu * num_halos_per_gpu
        end_gpu = start_gpu + num_halos_per_gpu
        M_chunk = M_chunk_all[start_gpu:end_gpu]
        ra_chunk = ra_chunk_all[start_gpu:end_gpu]
        dec_chunk = dec_chunk_all[start_gpu:end_gpu]
        z_chunk = z_chunk_all[start_gpu:end_gpu]
        vlos_chunk = vlos_chunk_all[start_gpu:end_gpu]

        scale_fac = 1. / (1. + z_chunk)
        rho_c_z = constants.RHO_CRIT_0_KPC3 * bkgrd.Esqr(B12_test.cosmo_jax, scale_fac) * 1e9
        rho_threshold = 200 * rho_c_z
        R200c = np.array((M_chunk * 3.0 / (4.0 * np.pi * rho_threshold))**(1.0 / 3.0))
        DA = np.array(bkgrd.angular_diameter_distance(B12_test.cosmo_jax, scale_fac))
        max_paint_R200c_factor = 3.0

        # Clip coordinates to valid ranges
        ra_chunk_clipped = np.clip(ra_chunk, 0.01, 359.99)
        dec_chunk_clipped = np.clip(dec_chunk, -89.99, 89.99)

        def process_halo_cpu(jhalo) -> Tuple[np.ndarray, ...]:
            """Process a single halo on CPU to find nearby pixels."""
            # (jhalo, ra_arr, dec_arr, r200c_arr, da_arr, m_arr, 
            # z_arr, vlos_arr, nside, max_paint_factor) = args
            
            # Find nearby pixels
            vec = hp.ang2vec(ra_chunk_clipped[jhalo], dec_chunk_clipped[jhalo], lonlat=True)
            nearby_angle = max_paint_R200c_factor * R200c[jhalo] / DA[jhalo]
            nearby_pix = hp.query_disc(nside, vec, nearby_angle)
            nearby_ra, nearby_dec = hp.pix2ang(nside, nearby_pix, lonlat=True)
            
            # Calculate physical distances using Haversine formula
            def haversine(theta):
                return np.sin(theta / 2.) ** 2.
            
            ra1, dec1 = np.deg2rad(ra_chunk_clipped[jhalo]), np.deg2rad(dec_chunk_clipped[jhalo])
            ra2, dec2 = np.deg2rad(nearby_ra), np.deg2rad(nearby_dec)
            theta = 2. * np.arcsin(np.sqrt(
                haversine(dec1 - dec2) + np.cos(dec1) * np.cos(dec2) * haversine(ra1 - ra2)
            ))
            physical_distances = DA[jhalo] * theta

            num_pix = len(nearby_pix)
            return (
                np.array(nearby_pix, dtype=np.int32),
                np.array(physical_distances, dtype=np.float64),
                np.full(num_pix, np.log(M_chunk[jhalo]), dtype=np.float64),
                np.full(num_pix, z_chunk[jhalo], dtype=np.float64),
                np.full(num_pix, vlos_chunk[jhalo], dtype=np.float64),
                np.full(num_pix, DA[jhalo], dtype=np.float64),
                np.full(num_pix, max_paint_R200c_factor * R200c[jhalo], dtype=np.float64)
            )

        # Process halos in parallel on CPU
        # if n_cpus is None:
        n_cpus = cpu_count()
        print(f"Using {n_cpus} CPU cores for preprocessing")
        with Pool(processes=n_cpus) as pool:
            # results = pool.map(process_halo_cpu, pool_args)
            results = pool.map(process_halo_cpu, range(len(z_chunk)))

        # Concatenate results
        (nearby_pix_all, distances_pix_all, start_ind_all, end_ind_all, 
            logM_ind_all, z_ind_all, vlos_ind_all, ang_distance_all, rp_max_all) = concatenate_halo_results(results)


        mock_params_dict = {
            'halo_z': jnp.array(z_chunk),
            'halo_ra': jnp.array(ra_chunk_clipped),
            'halo_dec': jnp.array(dec_chunk_clipped),
            'halo_M': jnp.array(M_chunk),
            'halo_vlos': jnp.array(vlos_chunk),
            'nearby_pix_all': jnp.array(nearby_pix_all),
            'pix_prop_all': jnp.array([
                np.log(distances_pix_all), z_ind_all, 
                logM_ind_all, vlos_ind_all
            ]).T,
            'start_ind': jnp.int32(start_ind_all),
            'end_ind': jnp.int32(end_ind_all),
            'ang_distance_all': jnp.array(ang_distance_all)[start_ind_all],
            'rp_max_all': jnp.array(rp_max_all)[start_ind_all],
            'nside': nside,
            'get_ymap': True,
            'smooth_profiles': True
        }
    
        mock_params_dict_all_gpus[jgpu] = mock_params_dict
        nelem_all.append(mock_params_dict['pix_prop_all'].shape[0])

    


Node 0:   0%|          | 0/1 [00:00<?, ?it/s]

0 100000
Using 64 CPU cores for preprocessing
Using 64 CPU cores for preprocessing


Node 0: 100%|██████████| 1/1 [00:05<00:00,  5.80s/it]


In [5]:
mock_params_dict_all_gpus[jgpu]['pix_prop_all'].shape


(2089239, 4)

In [14]:
nelem_all = np.array(nelem_all)
# print(np.amax(nelem_all))
nelem_max = np.amax(nelem_all)
for jgpu in range(n_gpus_local):
    nearby_pix_all = mock_params_dict_all_gpus[jgpu]['nearby_pix_all']
    pix_prop_all = mock_params_dict_all_gpus[jgpu]['pix_prop_all']
    if len(nearby_pix_all) < nelem_max:
        padding = nelem_max - len(nearby_pix_all)
        nearby_pix_all = np.pad(nearby_pix_all, (0, padding), mode='constant', constant_values=0)
        pix_prop_all = np.pad(pix_prop_all, ((0, padding), (0,0)), mode='constant', constant_values=0)
        mock_params_dict_all_gpus[jgpu]['nearby_pix_all'] = jnp.array(nearby_pix_all)
        mock_params_dict_all_gpus[jgpu]['pix_prop_all'] = jnp.array(pix_prop_all)





In [15]:
mock_params_dict = {}
for key, val in mock_params_dict_all_gpus[0].items():
    if key in ['start_ind', 'end_ind', 'ang_distance_all', 'rp_max_all',
                        'halo_z', 'halo_ra', 'halo_dec', 'halo_M', 'halo_vlos',
                        'nearby_pix_all', 'pix_prop_all']:

        # concatenate the arrays along the first axis
        mock_params_dict[key] = jnp.stack([mock_params_dict_all_gpus[jgpu][key] for jgpu in range(n_gpus_local)], axis=0)
    else:
        mock_params_dict[key] = mock_params_dict_all_gpus[0][key]



In [16]:
# from get_sim_maps_pmap import get_sim_map
# # Define pmapped function for multi-GPU execution
# @partial(pmap, in_axes=(None, None, None, None, None, 0, 0, 0, 0, 0, 0),
#         static_broadcasted_argnums=(0, 1, 2, 3, 4))
# def get_sim_map_pmap(sim_params, halo_params, analysis, other_params, profile, mock_params):
#     return get_sim_map(sim_params, halo_params, analysis, other_params, profile, mock_params)



In [17]:
analysis_dict['get_ymap'] = True
analysis_dict['smooth_profiles'] = True
analysis_dict['nside'] = 1024


In [18]:
# mock_maps_gpus = get_sim_map_pmap(
#             sim_params_dict, halo_params_dict, analysis_dict,
#             other_params_dict, mock_params_dict, B12_test
#         )
from get_sim_maps_pmap import get_sim_map
def _pmap_wrapper(start_ind,
                end_ind,
                ang_distance_all,
                rp_max_all,
                nearby_pix_all,
                pix_prop_all):
    """Wrapper for get_sim_map for use with pmap."""
    return get_sim_map(
        sim_params_dict, halo_params_dict, analysis_dict,
        other_params_dict, B12_test, start_ind,
        end_ind,
        ang_distance_all,
        rp_max_all,
        nearby_pix_all,
        pix_prop_all
    )



In [19]:
pmapped_simulation = jax.pmap(_pmap_wrapper)
mock_maps_gpus = pmapped_simulation(mock_params_dict['start_ind'],
                                        mock_params_dict['end_ind'],
                                        mock_params_dict['ang_distance_all'],
                                        mock_params_dict['rp_max_all'],
                                        mock_params_dict['nearby_pix_all'],
                                        mock_params_dict['pix_prop_all'])




ConcretizationTypeError: Abstract tracer value encountered where concrete value is expected: traced array with shape int32[3479101]
The error arose for the first argument of jnp.unique(). To make jnp.unique() compatible with JIT and other transforms, you can specify a concrete value for the size argument, which will determine the output size.
The error occurred while tracing the function _pmap_wrapper at /tmp/ipykernel_1873416/2210924408.py:6 for pmap. This concrete value was not available in Python because it depends on the value of the argument nearby_pix_all.

See https://jax.readthedocs.io/en/latest/errors.html#jax.errors.ConcretizationTypeError

In [29]:
# def _pmap_wrapper(mock_params_for_device):
#             """Wrapper for get_sim_map for use with pmap."""
#             return get_sim_map(
#                 sim_params_dict, halo_params_dict, analysis_dict,
#                 other_params_dict, mock_params_for_device, B12_test
#             )

# # Apply pmap to our wrapper function. JAX can now handle this function
# # without encountering unhashable dictionaries.
# pmapped_simulation = jax.pmap(_pmap_wrapper)

# Execute on multiple GPUs by calling the new pmapped function.
# mock_maps_gpus = pmapped_simulation(mock_params_dict)
# ----------------------------------------------------------------------
# FIXED CODE BLOCK ENDS HERE

# Sum maps from all GPUs
chunk_map = jnp.sum(mock_maps_gpus.ymap_final, axis=0)



ValueError: pmap was requested to map its argument along axis 0, which implies that its rank should be at least 1, but is only 0 (its shape is ())

In [ ]:

    # Check if we can use multiple GPUs
    num_halos_chunk = len(mock_params_dict['halo_z'])
    print(f"Node {node_rank}: Processing {num_halos_chunk:,} halos in chunk {chunk_idx}")

    if n_gpus_local > 1 and num_halos_chunk >= n_gpus_local:
        # Ensure divisibility by number of GPUs
        num_halos_pmap = (num_halos_chunk // n_gpus_local) * n_gpus_local
        
        # Prepare data for pmap
        mock_params_pmap = {}
        for key, val in mock_params_dict.items():
            if key in ['start_ind', 'end_ind', 'ang_distance_all', 'rp_max_all',
                        'halo_z', 'halo_ra', 'halo_dec', 'halo_M', 'halo_vlos']:
                # Reshape for pmap: (n_gpus, halos_per_gpu)
                mock_params_pmap[key] = val[:num_halos_pmap].reshape(n_gpus_local, -1)
            else:
                # Broadcast to all GPUs
                mock_params_pmap[key] = val
        
        print('starting pmap execution')
        # Execute on multiple GPUs
        mock_maps_gpus = get_sim_map_pmap(
            sim_params_dict, halo_params_dict, analysis_dict,
            other_params_dict, mock_params_pmap, B12_test
        )
        
        # Sum maps from all GPUs
        chunk_map = jnp.sum(mock_maps_gpus.ymap_final, axis=0)
    else:
        # Single GPU execution
        mock_map = get_sim_map(
            sim_params_dict, halo_params_dict, analysis_dict,
            other_params_dict, mock_params_dict, B12_test
        )
        chunk_map = mock_map.ymap_final
    
    # Add to node's map
    map_node += np.array(np.nan_to_num(chunk_map), dtype=np.float32)
    
    # Periodic cleanup
    if chunk_idx % 5 == 0:
        jax.clear_caches()
        gc.collect()

# Save node's map
sdir = '/mnt/home/spandey/ceph/GODMAX/notebooks/all_arxiv/mock_gen/maps_halfdome/'
os.makedirs(sdir, exist_ok=True)
save_map_fname = sdir + f'tSZ_sim_B12_node_{node_rank}_{num_nodes}_nside_{nside}_zmax_{zmax}.pkl'

print(f"Node {node_rank}: Saving map to {save_map_fname}")
saved = {
    'map_node': map_node,
    'node_rank': node_rank,
    'num_nodes': num_nodes,
    'n_halos_processed': len(M200c_node),
    'nside': nside,
    'zmax': zmax
}

with open(save_map_fname, 'wb') as f:
    pk.dump(saved, f)

# Print summary
print(f"Node {node_rank} Summary:")
print(f"  Processed {len(M200c_node):,} halos")
print(f"  Map mean: {np.mean(map_node):.2e}, std: {np.std(map_node):.2e}")
print(f"  Non-zero pixels: {np.sum(map_node != 0):,}")

# Wait for all nodes to complete
jax.experimental.multihost_utils.sync_global_devices("complete")

if node_rank == 0:
    print(f"="*60)
    print("All nodes completed successfully!")
    print(f"="*60)

# if __name__ == "__main__":
    # main()



In [17]:
# Process chunks
for chunk_idx in tqdm(range(num_chunks), desc=f"Node {node_rank}", disable=(node_rank != 0)):
    # Get chunk slice
    start_idx = chunk_idx * chunk_size
    end_idx = min((chunk_idx + 1) * chunk_size, len(M200c_node))
    print(start_idx, end_idx)
    M_chunk = M200c_node[start_idx:end_idx]
    ra_chunk = ra_node[start_idx:end_idx]
    dec_chunk = dec_node[start_idx:end_idx]
    z_chunk = z_node[start_idx:end_idx]
    vlos_chunk = vlos_node[start_idx:end_idx]
    
    if len(M_chunk) == 0:
        continue
    
    # # Prepare chunk for GPU processing
    # mock_params_dict = prepare_chunk_for_gpu(
    #     M_chunk, ra_chunk, dec_chunk, z_chunk, vlos_chunk,
    #     B12_test, nside
    # )
    scale_fac = 1. / (1. + z_chunk)
    rho_c_z = constants.RHO_CRIT_0_KPC3 * bkgrd.Esqr(B12_test.cosmo_jax, scale_fac) * 1e9
    rho_threshold = 200 * rho_c_z
    R200c = np.array((M_chunk * 3.0 / (4.0 * np.pi * rho_threshold))**(1.0 / 3.0))
    DA = np.array(bkgrd.angular_diameter_distance(B12_test.cosmo_jax, scale_fac))
    max_paint_R200c_factor = 3.0

    # Clip coordinates to valid ranges
    ra_chunk_clipped = np.clip(ra_chunk, 0.01, 359.99)
    dec_chunk_clipped = np.clip(dec_chunk, -89.99, 89.99)

    def process_halo_cpu(jhalo) -> Tuple[np.ndarray, ...]:
        """Process a single halo on CPU to find nearby pixels."""
        # (jhalo, ra_arr, dec_arr, r200c_arr, da_arr, m_arr, 
        # z_arr, vlos_arr, nside, max_paint_factor) = args
        
        # Find nearby pixels
        vec = hp.ang2vec(ra_chunk_clipped[jhalo], dec_chunk_clipped[jhalo], lonlat=True)
        nearby_angle = max_paint_R200c_factor * R200c[jhalo] / DA[jhalo]
        nearby_pix = hp.query_disc(nside, vec, nearby_angle)
        nearby_ra, nearby_dec = hp.pix2ang(nside, nearby_pix, lonlat=True)
        
        # Calculate physical distances using Haversine formula
        def haversine(theta):
            return np.sin(theta / 2.) ** 2.
        
        ra1, dec1 = np.deg2rad(ra_chunk_clipped[jhalo]), np.deg2rad(dec_chunk_clipped[jhalo])
        ra2, dec2 = np.deg2rad(nearby_ra), np.deg2rad(nearby_dec)
        theta = 2. * np.arcsin(np.sqrt(
            haversine(dec1 - dec2) + np.cos(dec1) * np.cos(dec2) * haversine(ra1 - ra2)
        ))
        physical_distances = DA[jhalo] * theta

        num_pix = len(nearby_pix)
        return (
            np.array(nearby_pix, dtype=np.int32),
            np.array(physical_distances, dtype=np.float64),
            np.full(num_pix, np.log(M_chunk[jhalo]), dtype=np.float64),
            np.full(num_pix, z_chunk[jhalo], dtype=np.float64),
            np.full(num_pix, vlos_chunk[jhalo], dtype=np.float64),
            np.full(num_pix, DA[jhalo], dtype=np.float64),
            np.full(num_pix, max_paint_R200c_factor * R200c[jhalo], dtype=np.float64)
        )

    # Process halos in parallel on CPU
    # if n_cpus is None:
    n_cpus = cpu_count()
    print(f"Using {n_cpus} CPU cores for preprocessing")
    with Pool(processes=n_cpus) as pool:
        # results = pool.map(process_halo_cpu, pool_args)
        results = pool.map(process_halo_cpu, range(len(z_chunk)))

    # Concatenate results
    (nearby_pix_all, distances_pix_all, start_ind_all, end_ind_all, 
        logM_ind_all, z_ind_all, vlos_ind_all, ang_distance_all, rp_max_all) = concatenate_halo_results(results)


    mock_params_dict = {
        'halo_z': jnp.array(z_chunk),
        'halo_ra': jnp.array(ra_chunk_clipped),
        'halo_dec': jnp.array(dec_chunk_clipped),
        'halo_M': jnp.array(M_chunk),
        'halo_vlos': jnp.array(vlos_chunk),
        'nearby_pix_all': jnp.array(nearby_pix_all),
        'pix_prop_all': jnp.array([
            np.log(distances_pix_all), z_ind_all, 
            logM_ind_all, vlos_ind_all
        ]).T,
        'start_ind': jnp.int32(start_ind_all),
        'end_ind': jnp.int32(end_ind_all),
        'ang_distance_all': jnp.array(ang_distance_all)[start_ind_all],
        'rp_max_all': jnp.array(rp_max_all)[start_ind_all],
        'nside': nside,
        'get_ymap': True,
        'smooth_profiles': True
    }
    
    print('created mock dict')

    if mock_params_dict is None:
        continue
    
    # Check if we can use multiple GPUs
    num_halos_chunk = len(mock_params_dict['halo_z'])
    print(f"Node {node_rank}: Processing {num_halos_chunk:,} halos in chunk {chunk_idx}")

    if n_gpus_local > 1 and num_halos_chunk >= n_gpus_local:
        # Ensure divisibility by number of GPUs
        num_halos_pmap = (num_halos_chunk // n_gpus_local) * n_gpus_local
        
        # Prepare data for pmap
        mock_params_pmap = {}
        for key, val in mock_params_dict.items():
            if key in ['start_ind', 'end_ind', 'ang_distance_all', 'rp_max_all',
                        'halo_z', 'halo_ra', 'halo_dec', 'halo_M', 'halo_vlos']:
                # Reshape for pmap: (n_gpus, halos_per_gpu)
                mock_params_pmap[key] = val[:num_halos_pmap].reshape(n_gpus_local, -1)
            else:
                # Broadcast to all GPUs
                mock_params_pmap[key] = val
        
        print('starting pmap execution')
        
        # FIXED CODE BLOCK STARTS HERE
        # ----------------------------------------------------------------------
        # Define a wrapper function that "closes over" the static, unhashable 
        # arguments (like dictionaries). This wrapper only takes the data that will be 
        # split across devices (mock_params_pmap) as an argument.
        def _pmap_wrapper(mock_params_for_device):
            """Wrapper for get_sim_map for use with pmap."""
            return get_sim_map(
                sim_params_dict, halo_params_dict, analysis_dict,
                other_params_dict, mock_params_for_device, B12_test
            )

        # Apply pmap to our wrapper function. JAX can now handle this function
        # without encountering unhashable dictionaries.
        pmapped_simulation = jax.pmap(_pmap_wrapper)
        
        # Execute on multiple GPUs by calling the new pmapped function.
        mock_maps_gpus = pmapped_simulation(mock_params_pmap)
        # ----------------------------------------------------------------------
        # FIXED CODE BLOCK ENDS HERE

        # Sum maps from all GPUs
        chunk_map = jnp.sum(mock_maps_gpus.ymap_final, axis=0)
    else:
        # Single GPU execution
        mock_map = get_sim_map(
            sim_params_dict, halo_params_dict, analysis_dict,
            other_params_dict, mock_params_dict, B12_test
        )
        chunk_map = mock_map.ymap_final
    
    # Add to node's map
    map_node += np.array(np.nan_to_num(chunk_map), dtype=np.float32)
    
    # Periodic cleanup
    if chunk_idx % 5 == 0:
        jax.clear_caches()
        gc.collect()

# Save node's map
sdir = '/mnt/home/spandey/ceph/GODMAX/notebooks/all_arxiv/mock_gen/maps_halfdome/'
os.makedirs(sdir, exist_ok=True)
save_map_fname = sdir + f'tSZ_sim_B12_node_{node_rank}_{num_nodes}_nside_{nside}_zmax_{zmax}.pkl'

print(f"Node {node_rank}: Saving map to {save_map_fname}")
saved = {
    'map_node': map_node,
    'node_rank': node_rank,
    'num_nodes': num_nodes,
    'n_halos_processed': len(M200c_node),
    'nside': nside,
    'zmax': zmax
}

with open(save_map_fname, 'wb') as f:
    pk.dump(saved, f)

# Print summary
print(f"Node {node_rank} Summary:")
print(f"  Processed {len(M200c_node):,} halos")
print(f"  Map mean: {np.mean(map_node):.2e}, std: {np.std(map_node):.2e}")
print(f"  Non-zero pixels: {np.sum(map_node != 0):,}")

# Wait for all nodes to complete
jax.experimental.multihost_utils.sync_global_devices("complete")

if node_rank == 0:
    print(f"="*60)
    print("All nodes completed successfully!")
    print(f"="*60)

    

Node 0:   0%|          | 0/32 [00:00<?, ?it/s]

0 100000
Using 64 CPU cores for preprocessing


Node 0:   0%|          | 0/32 [00:04<?, ?it/s]

created mock dict
Node 0: Processing 100,000 halos in chunk 0
starting pmap execution


ValueError: pmap was requested to map its argument along axis 0, which implies that its rank should be at least 1, but is only 0 (its shape is ())

In [23]:
mock_params_pmap['nearby_pix_all'].shape, mock_params_pmap['halo_z'].shape


((5568340,), (2, 50000))

In [21]:
mock_params_pmap.keys()


dict_keys(['halo_z', 'halo_ra', 'halo_dec', 'halo_M', 'halo_vlos', 'nearby_pix_all', 'pix_prop_all', 'start_ind', 'end_ind', 'ang_distance_all', 'rp_max_all', 'nside', 'get_ymap', 'smooth_profiles'])